# Evaluacion del modelo base

Carga el checkpoint guardado por `src.train` y genera todas las figuras y reportes que el documento final exige:

- Curvas de loss y accuracy por epoca.
- Matriz de confusion del conjunto de prueba.
- Reporte por clase (precision, recall, F1).
- Casos mal clasificados (analisis cualitativo).
- Comparativa del tamano de los exports (.keras, .tflite, .tflite int8).

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import confusion_matrix, classification_report

from src.config import PROCESSED_DIR, MODELS_DIR, CLASSES

print(f'tensorflow {tf.__version__}')
print(f'MODELS_DIR: {MODELS_DIR}')

## 1. Curvas de entrenamiento

In [ ]:
history_path = MODELS_DIR / 'training_history.json'
history = json.loads(history_path.read_text())
epochs = np.arange(1, len(history['loss']) + 1)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(epochs, history['loss'], label='train'); ax[0].plot(epochs, history.get('val_loss', []), label='val')
ax[0].set_title('Loss'); ax[0].set_xlabel('epoca'); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(epochs, history['accuracy'], label='train'); ax[1].plot(epochs, history.get('val_accuracy', []), label='val')
ax[1].set_title('Accuracy'); ax[1].set_xlabel('epoca'); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig(MODELS_DIR / 'training_curves.png', dpi=120); plt.show()

## 2. Reporte y matriz de confusion en test

In [ ]:
report = json.loads((MODELS_DIR / 'training_report.json').read_text())
print(f"Tiempo entrenamiento: {report['elapsed_seconds']:.1f}s   epocas corridas: {report['epochs_run']}")
for split, info in report['splits'].items():
    if info.get('n', 0) == 0: continue
    print(f"\n[{split}]  n={info['n']}  accuracy={info['accuracy']:.4f}")
    for cls, m in info['per_class'].items():
        print(f"   {cls:14s} P={m['precision']:.3f}  R={m['recall']:.3f}  F1={m['f1']:.3f}  n={m['support']}")

In [ ]:
cm = np.array(report['splits']['test']['confusion_matrix'])
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(CLASSES))); ax.set_xticklabels(CLASSES, rotation=45, ha='right')
ax.set_yticks(range(len(CLASSES))); ax.set_yticklabels(CLASSES)
ax.set_xlabel('Predicho'); ax.set_ylabel('Real'); ax.set_title('Matriz de confusion - test')
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.colorbar(im, ax=ax); plt.tight_layout()
plt.savefig(MODELS_DIR / 'confusion_matrix.png', dpi=120); plt.show()

## 3. Verificacion del modelo cargado y prediccion sobre test

In [ ]:
model = tf.keras.models.load_model(MODELS_DIR / 'cnn_voz_base.keras')
model.summary()

In [ ]:
data = np.load(PROCESSED_DIR / 'dataset.npz')
X_test, y_test = data['X_test'], data['y_test']
probs = model.predict(X_test, verbose=0)
y_pred = probs.argmax(axis=1)
acc = (y_pred == y_test).mean()
print(f'Test accuracy (recomputado): {acc:.4f}')
print(classification_report(y_test, y_pred, labels=list(range(len(CLASSES))),
                            target_names=CLASSES, zero_division=0))

## 4. Errores en test (analisis cualitativo)

In [ ]:
meta = json.loads((PROCESSED_DIR / 'dataset.meta.json').read_text())
errors = []
for i, (yt, yp) in enumerate(zip(y_test, y_pred)):
    if yt != yp:
        errors.append({
            'idx': int(i),
            'real': CLASSES[yt],
            'predicho': CLASSES[yp],
            'confianza': float(probs[i, yp]),
            'file': meta['test'][i]['file'],
            'env': meta['test'][i]['env'],
        })
print(f'Errores: {len(errors)}/{len(y_test)}')
for e in errors:
    print(f"   real={e['real']:<12} pred={e['predicho']:<12} conf={e['confianza']:.3f}"
          f"   ({e['env']})  {e['file']}")

## 5. Tamaños de export

In [ ]:
for fname in ['cnn_voz_base.keras', 'cnn_voz_base.tflite', 'cnn_voz_base_int8.tflite']:
    p = MODELS_DIR / fname
    if p.exists():
        print(f'{fname:32s} {p.stat().st_size/1024:>8.1f} KB')
    else:
        print(f'{fname:32s}  (no existe)')